In [9]:
# Import essential libraries for authentication
# Spotipy is the go-to Python wrapper for the Spotify API
# Spotify uses OAuth 2.0, so we will import that
import spotipy
from spotipy.oauth2 import SpotifyOAuth
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Used to hide credentials
from dotenv import load_dotenv
import os

# FastAPI import
from fastapi import APIRouter

In [10]:
# Authentication via spotipy.Spotify()
load_dotenv()
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=os.getenv("SPOTIFY_CLIENT_ID"),
    client_secret=os.getenv("SPOTIFY_CLIENT_SECRET"),
    redirect_uri=os.getenv("SPOTIFY_REDIRECT_URI"),
    scope="user-library-read user-top-read user-read-recently-played"
))

## Step 1: Pull Listening Data
Fetch liked songs, top tracks (short + medium term), and recently played.

In [ ]:
# Pull and store music data from multiple sources
# i.e. Liked Songs, Top Tracks/Artists, and Recently Played"
# Starting with Liked Songs
def get_liked_songs(sp):
    results = sp.current_user_saved_tracks(limit=50),
    tracks = []
    while results:
        for item in results['items']:
            track = item['track']
            tracks.append({
                'id': track['id'],
                'name': track['name'],
                'artist': track['artists'][0]['name'],
                'artist_id': track['artists'][0]['id'],
                'popularity': track['popularity'],
                'source': 'liked'
        })
        results = sp.next(results)  # handles pagination
    return tracks

# Top Tracks
# time_range options:
# - 'short_term'  = last 4 weeks
# - 'medium_term' = last 6 months (default)
# - 'long_term'   = all time
def get_top_tracks(sp, time_range='medium_term', limit=50):
    results = sp.current_user_top_tracks(
        time_range=time_range,
        limit=limit
    )
    tracks = []
    for item in results['items']:
        tracks.append({
            'id': item['id'],
            'name': item['name'],
            'artist': item['artists'][0]['name'],
            'artist_id': item['artists'][0]['id'],
            'popularity': item['popularity'],
            'source': f'top_tracks_{time_range}'
        })
    return tracks

# Top artists
# Useful for seeding recommendations
# and filtering candidates by genre
def get_top_artists(sp, time_range='medium_term', limit=50):
    results = sp.current_user_top_artists(
        time_range=time_range,
        limit=limit
    )
    artists = []
    for item in results['items']:
        artists.append({
            'id': item['id'],
            'name': item['name'],
            'genres': item['genres'],
            'popularity': item['popularity'],
            'source': f'top_artists_{time_range}'
        })
    return artists

# Lastly, get recently played
def get_recently_played(sp, limit=50):
    results = sp.current_user_recently_played(limit=limit)
    tracks = []
    for item in results['items']:
        track = item['track']
        tracks.append({
            'id': track['id'],
            'name': track['name'],
            'artist': track['artists'][0]['name'],
            'artist_id': track['artists'][0]['id'],
            'played_at': item['played_at'],   # timestamp — useful for weighting
            'source': 'recently_played'
        })
    return tracks

# Combine liked songs, top tracks, and recently played
# To make a cohesive taste profile for the user
def get_full_taste_profile(sp):
    all_tracks = (
        get_liked_songs(sp) +
        get_top_tracks(sp, 'short_term') +
        get_top_tracks(sp, 'medium_term') +
        get_recently_played(sp)
    )
    seen = set()
    unique = []
    for t in all_tracks:
        if t['id'] not in seen:
            seen.add(t['id'])
            unique.append(t)
    return unique

tracks = get_full_taste_profile(sp)
print(f"Total unique tracks in taste profile: {len(tracks)}")

## Step 2: Audio Features
Fetch Spotify's audio features for each track and explore the distributions.

In [ ]:
# Spotify allows max 100 track IDs per request
def get_audio_features(tracks):
    track_ids = [t['id'] for t in tracks]
    features = []
    # batch into chunks of 100
    for i in range(0, len(track_ids), 100):
        batch = track_ids[i:i+100]
        results = sp.audio_features(batch)
        features.extend([f for f in results if f is not None])
    return features

features = get_audio_features(sp, tracks)
df = pd.DataFrame(features)
df.head()

In [ ]:
# Visualize feature distribution
feature_cols = ['danceability', 'energy', 'valence',
                'tempo', 'acousticness', 'instrumentalness', 'loudness']

df[feature_cols].hist(bins=20, figsize=(14, 8), color='steelblue', edgecolor='white')
plt.suptitle("Audio Feature Distributions of Your Taste Profile", fontsize=14)
plt.tight_layout()
plt.show()

## Step 3: Build User Taste Profile & Score Candidates
Average the audio features of all liked/top/recent tracks to create a single
taste vector, then score Spotify-generated candidates by cosine similarity.

In [3]:
# Recommender model
# Use values from previous functions in order to create model that will recommend songs and artists
def build_recommendations(sp, tracks, features, n=10):
    feature_cols = [
        'danceability', 'energy', 'valence',
        'tempo', 'acousticness', 'instrumentalness', 'loudness'
    ]

    df = pd.DataFrame(features)[feature_cols].dropna()
    scaler = StandardScaler()
    scaled = scaler.fit_transform(df)
    user_profile = scaled.mean(axis=0).reshape(1, -1)

    # get top artists to use as seeds alongside tracks
    top_artists = get_top_artists(sp, limit=50)

    # mix track and artist seeds (max 5 total combined)
    seed_tracks = [t['id'] for t in tracks[:2]]
    seed_artists = [a['id'] for a in top_artists[:3]]

    candidates = sp.recommendations(
        seed_tracks=seed_tracks,
        seed_artists=seed_artists,
        limit=50
    )

    candidate_ids = [t['id'] for t in candidates['tracks']]
    candidate_features = sp.audio_features(candidate_ids)
    candidate_df = pd.DataFrame(
        [f for f in candidate_features if f is not None]
    )[feature_cols].dropna()
    candidate_scaled = scaler.transform(candidate_df)

    similarities = cosine_similarity(user_profile, candidate_scaled)[0]
    top_indices = similarities.argsort()[::-1][:n]

    recommendations = []
    for i in top_indices:
        track = candidates['tracks'][i]
        recommendations.append({
            'name': track['name'],
            'artist': track['artists'][0]['name'],
            'id': track['id'],
            'similarity_score': round(float(similarities[i]), 4)
        })
    return recommendations

In [ ]:
# API Endpoints
# Want to recommend songs based on the user's full taste profile 
# i.e. Liked Songs + Top Tracks + Recently Played combined